In [ ]:
!pip install -q -U transformers peft bitsandbytes accelerate datasets trl scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.8/838.8 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 10.2 MB/s eta 0:00:00


In [ ]:
!wget https://zenodo.org/records/3713280/files/pan12-sexual-predator-identification-test-and-training.zip?download=1 -O pan12-sexual-predator-identification-test-and-training.zip
!unzip pan12-sexual-predator-identification-test-and-training.zip

--2026-06-30 08:23:27--  https://zenodo.org/records/3713280/files/pan12-sexual-predator-identification-test-and-training.zip?download=1
Resolving zenodo.org (zenodo.org)... 188.184.98.114, 137.138.153.219, 188.185.48.75, ...
Connecting to zenodo.org (zenodo.org)|188.184.98.114|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 91163031 (87M) [application/octet-stream]
Saving to: ‘pan12-sexual-predator-identification-test-and-training.zip’

pan12-sexual-predat 100%[===================>]  86.94M  2.51MB/s    in 56s     

2026-06-30 08:24:24 (1.56 MB/s) - ‘pan12-sexual-predator-identification-test-and-training.zip’ saved [91163031/91163031]

Archive:  pan12-sexual-predator-identification-test-and-training.zip
  inflating: pan12-sexual-predator-identification-test-corpus-2012-05-21.zip  
  inflating: pan12-sexual-predator-identification-training-corpus-2012-05-01.zip  


In [ ]:
!unzip pan12-sexual-predator-identification-training-corpus-2012-05-01.zip
!unzip pan12-sexual-predator-identification-test-corpus-2012-05-21.zip

Archive:  pan12-sexual-predator-identification-training-corpus-2012-05-01.zip
  inflating: pan12-sexual-predator-identification-training-corpus-2012-05-01/readme.txt  
  inflating: pan12-sexual-predator-identification-training-corpus-2012-05-01/pan12-sexual-predator-identification-diff.txt  
  inflating: pan12-sexual-predator-identification-training-corpus-2012-05-01/pan12-sexual-predator-identification-training-corpus-2012-05-01.xml  
  inflating: pan12-sexual-predator-identification-training-corpus-2012-05-01/pan12-sexual-predator-identification-training-corpus-predators-2012-05-01.txt  
Archive:  pan12-sexual-predator-identification-test-corpus-2012-05-21.zip
  inflating: pan12-sexual-predator-identification-test-corpus-2012-05-21/pan12-sexual-predator-identification-test-corpus-2012-05-17.xml  
  inflating: pan12-sexual-predator-identification-test-corpus-2012-05-21/pan12-sexual-predator-identification-groundtruth-readme.txt  
  inflating: pan12-sexual-predator-identification-test-

In [ ]:
!mkdir -p pan2012
!mv pan12-sexual-predator-identification-training-corpus-2012-05-01 pan2012/train
!mv pan12-sexual-predator-identification-test-corpus-2012-05-21 pan2012/test

In [ ]:
from huggingface_hub import login
from google.colab import userdata

login(userdata.get('HF_TOKEN'))

In [8]:
"""
Detecting Online Grooming By Simple Contrastive Chat Embeddings
================================================================
Full pipeline implementation based on:
  Rezaee Borj, Raja, Bours — IWSPA '23

Pipeline:
  1. Data loading & preprocessing (PAN 2012 XML format)
  2. SimCSE sentence embeddings (RoBERTa / BERT variants)
  3. SVM / RF / NB / SGD classifiers
  4. Fusion: Sum, Product, Majority Voting (score- and decision-level)
  5. Evaluation with Accuracy, Precision, Recall, F1, F0.5

Colab setup (run once in a Colab cell before this script):
  !pip install transformers sentence-transformers scikit-learn torch

Expected PAN 2012 directory layout:
  pan2012/
    train/
      pan12-sexual-predator-identification-training-corpus-2012-05-01.xml
      pan12-sexual-predator-identification-training-corpus-predators-2012-05-01.txt
    test/
      pan12-sexual-predator-identification-test-corpus-2012-05-17.xml
      pan12-sexual-predator-identification-groundtruth-problem1.xml   (or .txt)
"""

import os
import re
import xml.etree.ElementTree as ET
from pathlib import Path
from typing import List, Tuple, Dict, Optional
import numpy as np
from scipy.stats import mode as scipy_mode
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, fbeta_score, classification_report
)
from sklearn.preprocessing import StandardScaler
from sentence_transformers import SentenceTransformer

import time
import json
from datetime import datetime
import joblib

# ─────────────────────────────────────────────────────────────────────────────
# 1. CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────

from google.colab import drive
drive.mount('/content/drive')

# Paths — adjust to match your Colab file locations
DATA_DIR = Path("pan2012")
TRAIN_XML       = DATA_DIR / "train" / "pan12-sexual-predator-identification-training-corpus-2012-05-01.xml"
TRAIN_PRED_TXT  = DATA_DIR / "train" / "pan12-sexual-predator-identification-training-corpus-predators-2012-05-01.txt"
TEST_XML        = DATA_DIR / "test"  / "pan12-sexual-predator-identification-test-corpus-2012-05-17.xml"
TEST_PRED_TXT   = DATA_DIR / "test"  / "pan12-sexual-predator-identification-groundtruth-problem1.txt"

# Checkpoint directory — embeddings and trained classifiers are cached here
CHECKPOINT_DIR = Path("/content/drive/MyDrive/grooming_detection/checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# SimCSE model names (all publicly available on HuggingFace via sentence-transformers)
SIMCSE_MODELS: Dict[str, str] = {
    "SimCSE-Base-Bert":    "princeton-nlp/sup-simcse-bert-base-uncased",
    "SimCSE-Large-Bert":   "princeton-nlp/sup-simcse-bert-large-uncased",
    "SimCSE-Base-RoBERTa": "princeton-nlp/sup-simcse-roberta-base",
    "SimCSE-Large-RoBERTa":"princeton-nlp/sup-simcse-roberta-large",
}

# Preprocessing thresholds (Section 3.1 of the paper)
MIN_MESSAGES   = 7   # conversations with fewer messages are discarded
MIN_AUTHORS    = 2   # conversations need exactly 2 authors
MAX_AUTHORS    = 2
RANDOM_STATE   = 42
BATCH_SIZE     = 64  # adjust down if GPU OOM

# ─────────────────────────────────────────────────────────────────────────────
# 2. CHECKPOINT HELPERS
# ─────────────────────────────────────────────────────────────────────────────

def _emb_cache_path(split: str, emb_key: str) -> Path:
    """Return path for cached numpy embeddings."""
    safe_key = emb_key.replace(" ", "_").replace("/", "-")
    return CHECKPOINT_DIR / f"embeddings_{split}_{safe_key}.npy"


def _clf_cache_path(emb_key: str, clf_key: str) -> Path:
    """Return path for a cached (scaler, clf) joblib bundle."""
    safe = emb_key.replace(" ", "_").replace("/", "-")
    return CHECKPOINT_DIR / f"clf_{safe}_{clf_key}.joblib"


def _meta_path() -> Path:
    return CHECKPOINT_DIR / "run_meta.json"


def save_embeddings(split: str, emb_key: str, embeddings: np.ndarray) -> None:
    path = _emb_cache_path(split, emb_key)
    np.save(path, embeddings)
    print(f"  [ckpt] Saved {split} embeddings → {path}")


def load_embeddings(split: str, emb_key: str) -> Optional[np.ndarray]:
    path = _emb_cache_path(split, emb_key)
    if path.exists():
        arr = np.load(path)
        print(f"  [ckpt] Loaded {split} embeddings from cache ({arr.shape})")
        return arr
    return None


def save_classifier(emb_key: str, clf_key: str,
                    scaler: StandardScaler, clf) -> None:
    path = _clf_cache_path(emb_key, clf_key)
    joblib.dump({"scaler": scaler, "clf": clf}, path)
    print(f"  [ckpt] Saved classifier → {path}")


def load_classifier(emb_key: str, clf_key: str) -> Optional[Tuple]:
    path = _clf_cache_path(emb_key, clf_key)
    if path.exists():
        bundle = joblib.load(path)
        print(f"  [ckpt] Loaded classifier from cache: {emb_key} + {clf_key}")
        return bundle["scaler"], bundle["clf"]
    return None


def save_run_meta(meta: dict) -> None:
    with open(_meta_path(), "w") as f:
        json.dump(meta, f, indent=2)


def load_run_meta() -> dict:
    if _meta_path().exists():
        with open(_meta_path()) as f:
            return json.load(f)
    return {}

# ─────────────────────────────────────────────────────────────────────────────
# 3. DATA LOADING & PREPROCESSING
# ─────────────────────────────────────────────────────────────────────────────

# Load predator IDs — try .txt first, fall back to .xml ground-truth
def load_preds(txt_path: Path) -> set:
    if txt_path.exists():
        return load_predator_ids(txt_path)
    xml_path = txt_path.with_suffix(".xml")
    if xml_path.exists():
        return load_predator_ids_from_xml(xml_path)
    raise FileNotFoundError(
        f"Cannot find predator ID file at {txt_path} or {xml_path}"
    )

def load_predator_ids(path: Path) -> set:
    """Load the set of known predator author-IDs from a plain-text file."""
    predators = set()
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            pid = line.strip()
            if pid:
                predators.add(pid)
    print(f"  Loaded {len(predators):,} predator IDs from {path.name}")
    return predators


def load_predator_ids_from_xml(path: Path) -> set:
    """
    Alternative: load predator IDs from the ground-truth XML used in some
    PAN 2012 test-set distributions.
    <users><user id="..."/></users>
    """
    tree = ET.parse(path)
    root = tree.getroot()
    return {u.get("id") for u in root.iter("user") if u.get("id")}


def is_english(text: str) -> bool:
    """Rough English-language filter: >80 % of chars are ASCII."""
    if not text:
        return False
    ascii_count = sum(1 for c in text if ord(c) < 128)
    return ascii_count / len(text) > 0.80


def clean_text(text: str) -> str:
    """
    Light cleaning per Section 3.1:
      - no stemming / lemmatisation (preserve subword information)
      - remove non-English tokens (words) but keep punctuation
    """
    # Remove XML-escaped artifacts
    text = text.replace("&amp;", "&").replace("&lt;", "<").replace("&gt;", ">")
    # Keep words that are mostly ASCII; drop fully non-ASCII tokens
    tokens = text.split()
    filtered = []
    for tok in tokens:
        ascii_ratio = sum(1 for c in tok if ord(c) < 128) / max(len(tok), 1)
        if ascii_ratio >= 0.70:
            filtered.append(tok)
    return " ".join(filtered).strip()


def load_conversations(xml_path: Path, predator_ids: set) -> Tuple[List[str], List[int]]:
    """
    Parse the PAN 2012 XML corpus and return (texts, labels).

    XML structure:
        <conversations>
          <conversation id="...">
            <message line="1">
              <author>...</author>
              <time>...</time>
              <text>...</text>
            </message>
            ...
          </conversation>
        </conversations>

    Labelling rule (Section 3.1):
        If ANY author in a conversation is in predator_ids → label = 1 (predatory)
        Otherwise → label = 0 (non-predatory)

    Filtering rules:
        - Discard conversations with < MIN_MESSAGES messages
        - Discard conversations with != 2 unique authors
        - Remove non-English content
    """
    print(f"  Parsing {xml_path} …")
    tree = ET.parse(xml_path)
    root = tree.getroot()

    texts, labels = [], []
    total = skipped_msg = skipped_authors = skipped_english = 0

    for conv in root.iter("conversation"):
        total += 1
        messages = list(conv.iter("message"))

        # Filter: too few messages
        if len(messages) < MIN_MESSAGES:
            skipped_msg += 1
            continue

        # Collect authors
        authors = {m.find("author").text.strip()
                   for m in messages
                   if m.find("author") is not None and m.find("author").text}

        # Filter: not exactly 2 authors
        if not (MIN_AUTHORS <= len(authors) <= MAX_AUTHORS):
            skipped_authors += 1
            continue

        # Merge all message texts into one document
        raw_parts = []
        for m in messages:
            t = m.find("text")
            if t is not None and t.text:
                raw_parts.append(t.text.strip())
        full_text = " ".join(raw_parts)
        cleaned   = clean_text(full_text)

        # Filter: not English enough
        if not is_english(cleaned) or len(cleaned) < 10:
            skipped_english += 1
            continue

        # Label
        label = 1 if authors & predator_ids else 0
        texts.append(cleaned)
        labels.append(label)

    print(f"    Total conversations : {total:,}")
    print(f"    Skipped (<{MIN_MESSAGES} msgs)   : {skipped_msg:,}")
    print(f"    Skipped (authors≠2) : {skipped_authors:,}")
    print(f"    Skipped (non-EN)    : {skipped_english:,}")
    print(f"    Kept → {len(texts):,}  "
          f"(predatory={sum(labels):,}, non-predatory={len(labels)-sum(labels):,})")
    return texts, labels


# ─────────────────────────────────────────────────────────────────────────────
# 4. SIMCSE SENTENCE EMBEDDINGS
# ─────────────────────────────────────────────────────────────────────────────

def get_embeddings(emb_key, model_name, split, texts,
                   device="cuda", force_recompute=False,
                   batch_size=BATCH_SIZE):
    """
    Return embeddings for `texts`, loading from cache when available.

    Args:
        emb_key:         Short name used as the cache key (e.g. "SimCSE-Base-Bert")
        model_name:      HuggingFace model path
        split:           "train" or "test"
        texts:           List of raw text strings to encode
        device:          "cuda" or "cpu"
        force_recompute: If True, ignore any existing cache and re-encode
    """
    if not force_recompute:
        cached = load_embeddings(split, emb_key)
        if cached is not None:
            return cached

    print(f"  Loading model: {model_name}")
    model = SentenceTransformer(model_name, device=device)
    print(f"  Encoding {len(texts):,} {split} texts …")
    embeddings = model.encode(
        texts,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=False,
    )
    print(f"  Embedding shape: {embeddings.shape}")

    save_embeddings(split, emb_key, embeddings)
    return embeddings


# ─────────────────────────────────────────────────────────────────────────────
# 5. CLASSIFIERS
# ─────────────────────────────────────────────────────────────────────────────

def build_classifiers() -> Dict[str, object]:
    """Return the four classifier variants used in the paper."""
    return {
        "SVM": SVC(
            kernel="rbf",
            C=1.0,
            probability=True,   # needed for score-level fusion
            random_state=RANDOM_STATE,
            class_weight="balanced",
        ),
        "RandomForest": RandomForestClassifier(
            n_estimators=200,
            random_state=RANDOM_STATE,
            class_weight="balanced",
            n_jobs=-1,
        ),
        "NaiveBayes": GaussianNB(),
        "SGD": SGDClassifier(
            loss="modified_huber",  # supports predict_proba
            random_state=RANDOM_STATE,
            class_weight="balanced",
            n_jobs=-1,
        ),
    }


def train_and_predict(
    X_train: np.ndarray,
    y_train: List[int],
    X_test: np.ndarray,
    emb_key: str,
    clf_key: str,
    clf,
    force_retrain: bool = False,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Scale → fit → predict, with checkpoint save/load.

    If a saved (scaler, clf) bundle exists and force_retrain=False,
    the bundle is loaded and only inference is run.
    """
    if not force_retrain:
        bundle = load_classifier(emb_key, clf_key)
        if bundle is not None:
            scaler, clf = bundle
            X_te = scaler.transform(X_test)
            preds  = clf.predict(X_te)
            probas = clf.predict_proba(X_te)[:, 1]
            return preds, probas

    # Fresh training
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_train)
    X_te = scaler.transform(X_test)

    clf.fit(X_tr, y_train)
    preds  = clf.predict(X_te)
    probas = clf.predict_proba(X_te)[:, 1]

    save_classifier(emb_key, clf_key, scaler, clf)
    return preds, probas


# ─────────────────────────────────────────────────────────────────────────────
# 6. METRICS
# ─────────────────────────────────────────────────────────────────────────────

def evaluate(y_true: List[int], y_pred: np.ndarray,
             label: str = "") -> Dict[str, float]:
    """Compute Accuracy, Precision, Recall, F1, F0.5."""
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    f1   = f1_score(y_true, y_pred, zero_division=0)
    f05  = fbeta_score(y_true, y_pred, beta=0.5, zero_division=0)

    if label:
        print(f"\n{'─'*55}")
        print(f"  {label}")
        print(f"{'─'*55}")
        print(f"  Accuracy  : {acc:.4f}")
        print(f"  Precision : {prec:.4f}")
        print(f"  Recall    : {rec:.4f}")
        print(f"  F1        : {f1:.4f}")
        print(f"  F0.5      : {f05:.4f}")

    return dict(accuracy=acc, precision=prec, recall=rec, f1=f1, f05=f05)


# ─────────────────────────────────────────────────────────────────────────────
# 7. FUSION STRATEGIES
# ─────────────────────────────────────────────────────────────────────────────

def sum_rule_fusion(score_list: List[np.ndarray]) -> np.ndarray:
    """Score-level sum fusion (Eq. 8 in the paper)."""
    return np.sum(score_list, axis=0)


def product_rule_fusion(score_list: List[np.ndarray]) -> np.ndarray:
    """Score-level product fusion (Eq. 9)."""
    result = np.ones(len(score_list[0]))
    for s in score_list:
        result *= s
    return result


def majority_voting_fusion(decision_list: List[np.ndarray]) -> np.ndarray:
    """Decision-level majority voting (Eq. 10)."""
    stack = np.stack(decision_list, axis=1)   # (N, K)
    voted, _ = scipy_mode(stack, axis=1)
    return voted.flatten().astype(int)


def scores_to_labels(scores: np.ndarray, threshold: float = 0.5) -> np.ndarray:
    """Convert probability scores to binary labels."""
    # When scores are sums/products they are not bounded to [0,1];
    # we threshold at the midpoint of the actual range.
    mid = (scores.max() + scores.min()) / 2
    return (scores >= mid).astype(int)


# ─────────────────────────────────────────────────────────────────────────────
# 8. FULL PIPELINE
# ─────────────────────────────────────────────────────────────────────────────

def run_pipeline(
    device: str = "cuda",
    force_recompute_embeddings: bool = False,
    force_retrain_classifiers: bool = False,
):
    """
    End-to-end execution of the paper's full pipeline.

    Steps:
      A) Load & preprocess PAN 2012 data
      B) For each of 4 SimCSE embeddings:
           - Encode train / test
           - Train 4 classifiers
           - Collect predictions & probability scores
      C) Evaluate each (embedding × classifier) combination
      D) Apply score-level and decision-level fusion across classifiers
         per embedding (Table 4 in the paper)
      E) Apply fusion across ALL 16 (embedding × classifier) combinations
         (Table 5 in the paper)

    Flags:
        force_recompute_embeddings  — ignore cached .npy files and re-encode
        force_retrain_classifiers   — ignore cached .joblib files and re-train
    """
    run_meta = load_run_meta()
    run_start = datetime.now().isoformat()
    print(f"\nRun started: {run_start}")

    # ── A. Data ──────────────────────────────────────────────────────────────
    print("\n" + "═"*60)
    print("  STEP 1 — Loading PAN 2012 dataset")
    print("═"*60)

    print("\n[Train]")
    train_preds = load_preds(TRAIN_PRED_TXT)
    train_texts, train_labels = load_conversations(TRAIN_XML, train_preds)

    print("\n[Test]")
    test_preds = load_preds(TEST_PRED_TXT)
    test_texts, test_labels  = load_conversations(TEST_XML, test_preds)

    y_train = np.array(train_labels)
    y_test  = np.array(test_labels)

    # ── B & C. Embeddings + Classifiers ──────────────────────────────────────
    print("\n" + "═"*60)
    print("  STEP 2 — Extracting embeddings & training classifiers")
    print("═"*60)

    # Storage for fusion
    # all_probas[emb_key][clf_key] = np.ndarray of shape (N_test,)
    # all_preds [emb_key][clf_key] = np.ndarray of shape (N_test,)
    all_probas: Dict[str, Dict[str, np.ndarray]] = {}
    all_preds:  Dict[str, Dict[str, np.ndarray]] = {}
    all_results: Dict[str, Dict[str, Dict]] = {}

    for emb_key, model_name in SIMCSE_MODELS.items():
        print(f"\n{'━'*60}")
        print(f"  Embedding: {emb_key}")
        print(f"{'━'*60}")

        X_train = get_embeddings(
            emb_key, model_name, "train", train_texts,
            device=device, force_recompute=force_recompute_embeddings,
        )
        X_test = get_embeddings(
            emb_key, model_name, "test", test_texts,
            device=device, force_recompute=force_recompute_embeddings,
        )

        all_probas[emb_key]  = {}
        all_preds[emb_key]   = {}
        all_results[emb_key] = {}

        for clf_key, clf in build_classifiers().items():
            print(f"\n  → Classifier: {clf_key}")
            t0 = time.time()
            preds, probas = train_and_predict(
                X_train, y_train, X_test,
                emb_key, clf_key, clf,
                force_retrain=force_retrain_classifiers,
            )
            elapsed = time.time() - t0

            all_preds[emb_key][clf_key]  = preds
            all_probas[emb_key][clf_key] = probas

            label  = f"{emb_key} + {clf_key}"
            result = evaluate(y_test, preds, label=label)
            result["train_time_s"] = round(elapsed, 2)
            all_results[emb_key][clf_key] = result

            # ── per-(emb, clf) checkpoint ─────────────────────────────────
            run_meta[f"{emb_key}_{clf_key}"] = {
                "completed": True,
                "timestamp": datetime.now().isoformat(),
                **{k: v for k, v in result.items() if k != "train_time_s"},
            }
            save_run_meta(run_meta)


    # ── D. Fusion per embedding (Table 4) ────────────────────────────────────
    print("\n" + "═"*60)
    print("  STEP 3 — Fusion per embedding (Table 4)")
    print("═"*60)

    per_emb_fusion_results: Dict[str, Dict[str, Dict]] = {}

    for emb_key in SIMCSE_MODELS:
        per_emb_fusion_results[emb_key] = {}
        clf_keys = list(all_probas[emb_key].keys())

        score_list    = [all_probas[emb_key][k] for k in clf_keys]
        decision_list = [all_preds[emb_key][k]  for k in clf_keys]

        # Sum
        sum_scores  = sum_rule_fusion(score_list)
        sum_labels  = scores_to_labels(sum_scores)
        r = evaluate(y_test, sum_labels, label=f"{emb_key} | Sum Fusion")
        per_emb_fusion_results[emb_key]["Sum"] = r

        # Product
        prod_scores = product_rule_fusion(score_list)
        prod_labels = scores_to_labels(prod_scores)
        r = evaluate(y_test, prod_labels, label=f"{emb_key} | Product Fusion")
        per_emb_fusion_results[emb_key]["Product"] = r

        # Majority voting
        maj_labels = majority_voting_fusion(decision_list)
        r = evaluate(y_test, maj_labels, label=f"{emb_key} | Majority Voting")
        per_emb_fusion_results[emb_key]["Majority"] = r

    # ── E. Fusion across all 16 combinations (Table 5) ───────────────────────
    print("\n" + "═"*60)
    print("  STEP 4 — Fusion of all 16 configurations (Table 5)")
    print("═"*60)

    all_score_list    = []
    all_decision_list = []
    for emb_key in SIMCSE_MODELS:
        for clf_key in all_probas[emb_key]:
            all_score_list.append(all_probas[emb_key][clf_key])
            all_decision_list.append(all_preds[emb_key][clf_key])

    global_sum_labels     = scores_to_labels(sum_rule_fusion(all_score_list))
    global_product_labels = scores_to_labels(product_rule_fusion(all_score_list))
    global_majority_labels = majority_voting_fusion(all_decision_list)

    global_results = {}
    global_results["Sum"]      = evaluate(y_test, global_sum_labels,
                                           label="ALL configurations | Sum Rule")
    global_results["Product"]  = evaluate(y_test, global_product_labels,
                                           label="ALL configurations | Product Rule")
    global_results["Majority"] = evaluate(y_test, global_majority_labels,
                                           label="ALL configurations | Majority Voting")

    # ── Final summary ─────────────────────────────────────────────────────────
    run_meta["run_start"]    = run_start
    run_meta["run_end"]      = datetime.now().isoformat()
    run_meta["global_fusion"] = global_results

    save_run_meta(run_meta)
    _print_summary(all_results, per_emb_fusion_results, global_results)

    return {
        "per_model_per_clf": all_results,
        "per_model_fusion":  per_emb_fusion_results,
        "global_fusion":     global_results,
    }

# ─────────────────────────────────────────────────────────────────────────────
# 9. SUMMARY TABLE
# ─────────────────────────────────────────────────────────────────────────────

def _print_summary(per_model_per_clf, per_model_fusion, global_results):
    header = f"{'Config':<42} {'Acc':>6} {'Prec':>6} {'Rec':>6} {'F1':>6} {'F0.5':>6}"
    sep    = "─" * len(header)

    print("\n\n" + "═"*len(header))
    print("  RESULTS SUMMARY")
    print("═"*len(header))

    print(f"\n{'Single Classifier Results':^{len(header)}}")
    print(sep); print(header); print(sep)
    for emb, clf_dict in per_model_per_clf.items():
        for clf, m in clf_dict.items():
            print(f"{emb+' + '+clf:<42} {m['accuracy']:>6.4f} {m['precision']:>6.4f} "
                  f"{m['recall']:>6.4f} {m['f1']:>6.4f} {m['f05']:>6.4f}")

    print(f"\n{'Per-Embedding Fusion Results':^{len(header)}}")
    print(sep); print(header); print(sep)
    for emb, fuse_dict in per_model_fusion.items():
        for ftype, m in fuse_dict.items():
            print(f"{emb+' | '+ftype:<42} {m['accuracy']:>6.4f} {m['precision']:>6.4f} "
                  f"{m['recall']:>6.4f} {m['f1']:>6.4f} {m['f05']:>6.4f}")

    print(f"\n{'Global Fusion Results (all 16 configs)':^{len(header)}}")
    print(sep); print(header); print(sep)
    for ftype, m in global_results.items():
        print(f"{'All configs | '+ftype:<42} {m['accuracy']:>6.4f} {m['precision']:>6.4f} "
              f"{m['recall']:>6.4f} {m['f1']:>6.4f} {m['f05']:>6.4f}")
    print(sep + "\n")


# ─────────────────────────────────────────────────────────────────────────────
# 10. QUICK DEMO WITH SYNTHETIC DATA
# ─────────────────────────────────────────────────────────────────────────────

def run_demo(device: str = "cpu"):
    """Smoke-test with 50 synthetic sentences — no PAN 2012 data needed."""
    print("\n" + "═"*60)
    print("  DEMO MODE  (synthetic data, SimCSE-Base-RoBERTa + SVM)")
    print("═"*60)

    import random
    random.seed(42)

    predatory  = [
        "hey can we keep this just between us ok",
        "you seem really mature for your age",
        "do you have a camera on your phone",
        "lets meet up sometime when your parents arent home",
        "i bought you a gift do you want it",
    ]
    normal = [
        "what did you think of the homework assignment",
        "the match yesterday was amazing did you watch it",
        "can you recommend a good movie to watch",
        "i am going to the library to study",
        "lets meet at the park with the whole group",
    ]

    def make(templates, n):
        return [random.choice(templates) + " " + random.choice(templates)
                for _ in range(n)]

    n = 25
    train_texts  = make(predatory, n) + make(normal, n)
    train_labels = [1]*n + [0]*n
    test_texts   = make(predatory, n) + make(normal, n)
    test_labels  = [1]*n + [0]*n

    emb_key    = "SimCSE-Base-RoBERTa"
    model_name = SIMCSE_MODELS[emb_key]

    X_train = get_embeddings(emb_key, model_name, "demo_train", train_texts,
                              batch_size=16, device=device)
    X_test  = get_embeddings(emb_key, model_name, "demo_test",  test_texts,
                              batch_size=16, device=device)

    clf = SVC(kernel="rbf", probability=True,
              random_state=42, class_weight="balanced")
    preds, _ = train_and_predict(
        X_train, train_labels, X_test,
        emb_key, "SVM_demo", clf,
    )
    evaluate(np.array(test_labels), preds,
             label="Demo: SimCSE-Base-RoBERTa + SVM")
    print("\nDemo finished. Run run_pipeline() with PAN 2012 data next.\n")


# ─────────────────────────────────────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    import torch
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")
    print(f"Checkpoint directory: {CHECKPOINT_DIR.resolve()}")

    # ── Option A: resume / full pipeline ──────────────────────────────────────
    results = run_pipeline(
        device=device,
        force_recompute_embeddings=False,   # set True to re-encode from scratch
        force_retrain_classifiers=False,    # set True to ignore .joblib cache
    )

    print(results)

    # ── Option B: quick smoke-test ────────────────────────────────────────────
    # run_demo(device=device)

Mounted at /content/drive
Using device: cuda
Checkpoint directory: /content/drive/MyDrive/grooming_detection/checkpoints

Run started: 2026-06-30T08:26:50.279997

════════════════════════════════════════════════════════════
  STEP 1 — Loading PAN 2012 dataset
════════════════════════════════════════════════════════════

[Train]
  Loaded 142 predator IDs from pan12-sexual-predator-identification-training-corpus-predators-2012-05-01.txt
  Parsing pan2012/train/pan12-sexual-predator-identification-training-corpus-2012-05-01.xml …
    Total conversations : 66,927
    Skipped (<7 msgs)   : 50,064
    Skipped (authors≠2) : 7,389
    Skipped (non-EN)    : 7
    Kept → 9,467  (predatory=952, non-predatory=8,515)

[Test]
  Loaded 254 predator IDs from pan12-sexual-predator-identification-groundtruth-problem1.txt
  Parsing pan2012/test/pan12-sexual-predator-identification-test-corpus-2012-05-17.xml …
    Total conversations : 155,128
    Skipped (<7 msgs)   : 116,083
    Skipped (authors≠2) : 17

config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

...

  Embedding shape: (9467, 768)
  [ckpt] Saved train embeddings → /content/drive/MyDrive/grooming_detection/checkpoints/embeddings_train_SimCSE-Base-RoBERTa.npy
  Loading model: princeton-nlp/sup-simcse-roberta-base


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  Encoding 21,716 test texts …


Batches:   0%|          | 0/340 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [20]:
import joblib
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer

# ─────────────────────────────────────────────────────────────────────────────
# 1. CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────

CHECKPOINT_DIR = Path("/content/drive/MyDrive/grooming_detection/checkpoints")

EMB_KEY = "SimCSE-Base-Bert"
MODEL_NAME = "princeton-nlp/sup-simcse-bert-base-uncased"
CLF_KEY = "SVM"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ─────────────────────────────────────────────────────────────────────────────
# 2. DATA PREPROCESSING FUNCTIONS
# ─────────────────────────────────────────────────────────────────────────────

def clean_text(text: str) -> str:
    """Light cleaning matching Section 3.1 of the pipeline."""
    text = text.replace("&amp;", "&").replace("&lt;", "<").replace("&gt;", ">")
    tokens = text.split()
    filtered = []
    for tok in tokens:
        ascii_ratio = sum(1 for c in tok if ord(c) < 128) / max(len(tok), 1)
        if ascii_ratio >= 0.70:
            filtered.append(tok)
    return " ".join(filtered).strip()

# ─────────────────────────────────────────────────────────────────────────────
# 3. INFERENCE PIPELINE
# ─────────────────────────────────────────────────────────────────────────────

def run_conversation_inference(list_of_conversations: list):
    """
    Accepts a list where each element is a LIST of messages (strings) representing
    a complete chat log. Joins the messages, cleans them, and returns predictions.
    """
    safe_emb_key = EMB_KEY.replace(" ", "_").replace("/", "-")
    model_path = CHECKPOINT_DIR / f"clf_{safe_emb_key}_{CLF_KEY}.joblib"

    if not model_path.exists():
        raise FileNotFoundError(f"Could not find saved model at: {model_path}")

    print(f"Loading classifier bundle from: {model_path.name}")
    bundle = joblib.load(model_path)
    scaler = bundle["scaler"]
    clf = bundle["clf"]

    # Step 1: Combine the separate messages of each conversation into a single block
    # This precisely mimics the "full_text = ' '.join(raw_parts)" from your training pipeline
    merged_conversations = []
    for chat_history in list_of_conversations:
        full_text = " ".join(chat_history)
        cleaned_text = clean_text(full_text)
        merged_conversations.append(cleaned_text)

    # Step 2: Load the exact matching encoder and embed text
    print(f"Loading embedding model: {MODEL_NAME} on {DEVICE}...")
    encoder = SentenceTransformer(MODEL_NAME, device=DEVICE)

    print("Generating sentence embeddings...")
    embeddings = encoder.encode(
        merged_conversations,
        batch_size=len(merged_conversations),
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=False
    )

    # Step 3: Scale features using the training pipeline's scaler
    X_scaled = scaler.transform(embeddings)

    # Step 4: Run inference
    print("Running classification...")
    predictions = clf.predict(X_scaled)
    probabilities = clf.predict_proba(X_scaled)[:, 1]

    return predictions, probabilities

# ─────────────────────────────────────────────────────────────────────────────
# 4. EXECUTION
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    import torch

    # Each list here represents a SINGLE complete conversation session
    predatory_chat_session = [
        "hello there",
        "im just chilling online and looking for new people to talk to",
        "hey can we keep this just between us ok",
        "i feel like you understand me better than older people do",
        "you seem really mature for your age",
        "what do you look like anyway",
        "do you have a camera on your phone",
        "lets meet up sometime when your parents arent home",
        "i can drive over to your neighborhood this weekend",
        "i bought you a gift do you want it",
    ]

    normal_chat_session = [
        "hey what's up",
        "did you finish studying for the exam tomorrow",
        "what did you think of the homework assignment",
        "it was way too long but i think i got most of it right",
        "the match yesterday was amazing did you watch it",
        "yeah our team played so well in the second half",
        "can you recommend a good movie to watch tonight",
        "i am going to the library to study if you want to join",
        "sounds good i need to return some books anyway",
        "lets meet at the park with the whole group first",
    ]

    # We pack both full conversations into our test batch list
    all_test_sessions = [predatory_chat_session, normal_chat_session]

    # Run the combined model inference
    preds, scores = run_conversation_inference(all_test_sessions)

    print("\n" + "="*60)
    print("  INFERENCE RESULTS")
    print("="*60)

    # Process Conversation 1 (Suspicious Batch)
    label_1 = "[*] Flagged (Suspicious)" if preds[0] == 1 else "[+] Normal"
    print(f"[Conversation 1 - Suspicious Session]")
    print(f" Message Count : {len(predatory_chat_session)}")
    print(f" Decision      : {label_1}")
    print(f" Confidence    : {scores[0]:.4f}")

    # Process Conversation 2 (Normal Batch)
    print(f"\n{'─'*60}")
    label_2 = "[*] Flagged (Suspicious)" if preds[1] == 1 else "[+] Normal"
    print(f"[Conversation 2 - Normal Session]")
    print(f" Message Count : {len(normal_chat_session)}")
    print(f" Decision      : {label_2}")
    print(f" Confidence    : {scores[1]:.4f}")
    print("="*60 + "\n")

Loading classifier bundle from: clf_SimCSE-Base-Bert_SVM.joblib
Loading embedding model: princeton-nlp/sup-simcse-bert-base-uncased on cuda...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Generating sentence embeddings...
Running classification...

  INFERENCE RESULTS
[Conversation 1 - Suspicious Session]
 Message Count : 10
 Decision      : [*] Flagged (Suspicious)
 Confidence    : 0.6263

────────────────────────────────────────────────────────────
[Conversation 2 - Normal Session]
 Message Count : 10
 Decision      : [*] Flagged (Suspicious)
 Confidence    : 0.6726



In [ ]:
"""
WebSocket + Model Inference Threading Snippet
=============================================
Two threads run concurrently:
  Thread 1 (WSClientThread)   — connects to the backend WebSocket, receives
                                 raw text payloads, and pushes them onto an
                                 inference queue.
  Thread 2 (InferenceThread)  — pulls items off the queue, runs them through
                                 the grooming-detection model (SimCSE embeddings
                                 + trained SVM classifier), and dispatches
                                 results to an optional callback.

Usage:
  1.  Point BACKEND_WS_URL at your server (e.g. "ws://localhost:5000/ws").
  2.  Set BACKEND_PASSWORD to the matching X-Password / token value.
  3.  Run:  python ws_inference.py
  4.  The main thread stays alive; Ctrl-C triggers a clean shutdown.

Dependencies (install once in Colab):
  !pip install websocket-client sentence-transformers scikit-learn joblib torch
"""

import json
import logging
import queue
import threading
import time
from pathlib import Path
from typing import Callable, Optional

import joblib
import numpy as np
import websocket  # websocket-client
from sentence_transformers import SentenceTransformer

# ── Logging ──────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(name)s — %(message)s",
)
logger = logging.getLogger("ws_inference")

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION  — edit these to match your setup
# ─────────────────────────────────────────────────────────────────────────────

BACKEND_WS_URL   = "ws://localhost:5000/learner_stream"   # or wss://…
BACKEND_PASSWORD = "1234"

# Paths to the checkpoint artefacts produced by grooming_detection_colab.py
CHECKPOINT_DIR  = Path("checkpoints")
EMB_KEY         = "SimCSE-Base-RoBERTa"
CLF_KEY         = "SVM"
MODEL_NAME      = "princeton-nlp/sup-simcse-roberta-base"

INFERENCE_DEVICE = "cpu"     # "cuda" if GPU is available
RECONNECT_DELAY  = 5         # seconds between WebSocket reconnect attempts
QUEUE_MAXSIZE    = 256


# ─────────────────────────────────────────────────────────────────────────────
# MODEL LOADER
# ─────────────────────────────────────────────────────────────────────────────

class GroomingDetector:
    """
    Wraps the SimCSE encoder + trained SVM classifier.
    Loads the classifier from the checkpoint saved by the training pipeline.
    """

    def __init__(
        self,
        model_name: str = MODEL_NAME,
        emb_key: str    = EMB_KEY,
        clf_key: str    = CLF_KEY,
        checkpoint_dir: Path = CHECKPOINT_DIR,
        device: str     = INFERENCE_DEVICE,
    ):
        logger.info("Loading SimCSE encoder …")
        self.encoder = SentenceTransformer(model_name, device=device)

        clf_path = checkpoint_dir / f"clf_{emb_key.replace(' ','_').replace('/','_')}_{clf_key}.joblib"
        if not clf_path.exists():
            raise FileNotFoundError(
                f"Classifier checkpoint not found: {clf_path}\n"
                "Run the training pipeline first (grooming_detection_colab.py)."
            )
        bundle = joblib.load(clf_path)
        self.scaler = bundle["scaler"]
        self.clf    = bundle["clf"]
        logger.info(f"Classifier loaded from {clf_path}")

    def predict(self, text: str) -> dict:
        """
        Encode one conversation text and return a prediction dict:
            {
              "label":       0 | 1,           # 1 = predatory
              "probability": float,            # P(predatory)
              "text_preview": str              # first 80 chars
            }
        """
        emb   = self.encoder.encode([text], convert_to_numpy=True)
        emb_s = self.scaler.transform(emb)
        label = int(self.clf.predict(emb_s)[0])
        prob  = float(self.clf.predict_proba(emb_s)[0][1])
        return {
            "label":        label,
            "probability":  round(prob, 4),
            "text_preview": text[:80],
        }


# ─────────────────────────────────────────────────────────────────────────────
# THREAD 1 — WebSocket client
# ─────────────────────────────────────────────────────────────────────────────

class WSClientThread(threading.Thread):
    """
    Connects to the backend WebSocket and forwards received messages
    onto `inference_queue`.

    The server is expected to push JSON objects with at least a "text" key:
        {"text": "hey can we keep this just between us …", "conv_id": "abc123"}

    If the connection drops, this thread retries automatically every
    RECONNECT_DELAY seconds until `stop()` is called.
    """

    def __init__(
        self,
        url: str,
        password: str,
        inference_queue: queue.Queue,
    ):
        super().__init__(name="WSClientThread", daemon=True)
        self.url             = f"{url}?token={password}"
        self.inference_queue = inference_queue
        self._stop_event     = threading.Event()
        self._ws: Optional[websocket.WebSocketApp] = None

    # ── Public API ────────────────────────────────────────────────────────────

    def stop(self):
        self._stop_event.set()
        if self._ws:
            self._ws.close()

    # ── Internal handlers ─────────────────────────────────────────────────────

    def _on_open(self, ws):
        logger.info("WebSocket connected.")

    def _on_message(self, ws, raw_message):
        try:
            payload = json.loads(raw_message)
        except (json.JSONDecodeError, TypeError):
            # Treat raw string as plain conversation text
            payload = {"text": str(raw_message)}

        text = payload.get("text", "").strip()
        if not text:
            logger.debug("Received empty payload — skipping.")
            return

        # Attach any extra metadata the server sent
        item = {**payload, "text": text}

        try:
            self.inference_queue.put_nowait(item)
        except queue.Full:
            logger.warning("Inference queue full — dropping message.")

    def _on_error(self, ws, error):
        logger.error(f"WebSocket error: {error}")

    def _on_close(self, ws, code, msg):
        logger.info(f"WebSocket closed ({code}: {msg})")

    # ── Thread entry ──────────────────────────────────────────────────────────

    def run(self):
        while not self._stop_event.is_set():
            logger.info(f"Connecting to {self.url} …")
            self._ws = websocket.WebSocketApp(
                self.url,
                on_open=self._on_open,
                on_message=self._on_message,
                on_error=self._on_error,
                on_close=self._on_close,
            )
            self._ws.run_forever()

            if not self._stop_event.is_set():
                logger.info(f"Reconnecting in {RECONNECT_DELAY}s …")
                time.sleep(RECONNECT_DELAY)

        logger.info("WSClientThread stopped.")


# ─────────────────────────────────────────────────────────────────────────────
# THREAD 2 — Inference worker
# ─────────────────────────────────────────────────────────────────────────────

class InferenceThread(threading.Thread):
    """
    Pulls items from `inference_queue`, runs them through GroomingDetector,
    and calls `result_callback(item, prediction)`.

    The default callback just logs the result; replace it with your own
    handler (e.g. push to a database, send an alert, return over WebSocket).
    """

    def __init__(
        self,
        detector: GroomingDetector,
        inference_queue: queue.Queue,
        result_callback: Optional[Callable] = None,
    ):
        super().__init__(name="InferenceThread", daemon=True)
        self.detector        = detector
        self.inference_queue = inference_queue
        self.result_callback = result_callback or self._default_callback
        self._stop_event     = threading.Event()

    def stop(self):
        self._stop_event.set()

    @staticmethod
    def _default_callback(item: dict, prediction: dict):
        flag  = "[*] PREDATORY" if prediction["label"] == 1 else "[+] safe"
        prob  = prediction["probability"]
        conv  = item.get("conv_id", "—")
        preview = prediction["text_preview"]
        logger.info(f"[{flag}]  p={prob:.3f}  conv={conv}  text='{preview}…'")

    def run(self):
        logger.info("InferenceThread ready.")
        while not self._stop_event.is_set():
            try:
                item = self.inference_queue.get(timeout=0.5)
            except queue.Empty:
                continue

            try:
                prediction = self.detector.predict(item["text"])
                self.result_callback(item, prediction)
            except Exception as exc:
                logger.exception(f"Inference error: {exc}")
            finally:
                self.inference_queue.task_done()

        logger.info("InferenceThread stopped.")


# ─────────────────────────────────────────────────────────────────────────────
# MANAGER — ties both threads together
# ─────────────────────────────────────────────────────────────────────────────

class InferencePipeline:
    """
    Convenience wrapper that starts and stops both threads together.

    Example
    -------
    >>> def my_callback(item, pred):
    ...     if pred["label"] == 1:
    ...         alert_moderation_team(item["conv_id"])
    >>>
    >>> pipeline = InferencePipeline(result_callback=my_callback)
    >>> pipeline.start()
    >>> # … runs until Ctrl-C or pipeline.stop()
    """

    def __init__(
        self,
        ws_url: str         = BACKEND_WS_URL,
        password: str       = BACKEND_PASSWORD,
        result_callback     = None,
        queue_maxsize: int  = QUEUE_MAXSIZE,
    ):
        self._queue = queue.Queue(maxsize=queue_maxsize)

        logger.info("Initialising grooming detector …")
        self._detector = GroomingDetector()

        self._ws_thread  = WSClientThread(ws_url, password, self._queue)
        self._inf_thread = InferenceThread(
            self._detector, self._queue, result_callback
        )

    def start(self):
        logger.info("Starting pipeline …")
        self._inf_thread.start()
        self._ws_thread.start()

    def stop(self):
        logger.info("Stopping pipeline …")
        self._ws_thread.stop()
        self._inf_thread.stop()
        self._ws_thread.join(timeout=5)
        self._inf_thread.join(timeout=5)
        logger.info("Pipeline stopped.")

    def is_alive(self) -> bool:
        return self._ws_thread.is_alive() and self._inf_thread.is_alive()


# ─────────────────────────────────────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    # Optional: define a custom result handler
    def on_result(item: dict, prediction: dict):
        """Replace this with your own downstream logic."""
        flag = "PREDATORY" if prediction["label"] == 1 else "safe"
        print(
            f"[{flag}] p={prediction['probability']:.3f} "
            f"| conv={item.get('conv_id', '?')} "
            f"| {prediction['text_preview']!r}"
        )

    pipeline = InferencePipeline(
        ws_url=BACKEND_WS_URL,
        password=BACKEND_PASSWORD,
        result_callback=on_result,
    )

    pipeline.start()

    try:
        while pipeline.is_alive():
            time.sleep(1)
    except KeyboardInterrupt:
        print("\nShutting down …")
    finally:
        pipeline.stop()